# <font color='black'>Introduction to Machine Learning</font>

---

<img src="images/ipsa_logo.png" width="100" align="right">


> Year: **2025-2026**

> Contributors: **TO BE DECIDED**


## <font color='black'>Contents</font>

---
 
1. [Setting up the data](#Set-Up)
2. [A faire](#LinearRegression)
 * [Gradient Descent](#GD)
3. [A Faire](#LinearRegression2)
 * [Feature Scaling](#FeatureScaling)
 * [Gradient Descent](#GD2)
 * [Normal Equation](#NE)

Part 1:

## Set-Up
---
Let's first import the package we will need throughout this notebook.

In [11]:
#===================
# time wasted here: 5
# ==================

import numpy as np
import pandas as pd
import seaborn as sns
import time as t

import matplotlib.pyplot as plt 
#tell jupiter to display plots in the notebook.
%matplotlib inline 

Let's now import and visualize the data.

In [12]:
t0 = t.time()
data = pd.read_csv("./data/archive/Combined_Flights_2018.csv")
load_time_csv = t.time() - t0
t0 = t.time()
data = pd.read_parquet("./data/archive/Combined_Flights_2018.parquet")
load_time_parquet = t.time() - t0
print(f"shape of data: {data.shape}, load time for csv: {load_time_csv:.02f} sec, load time for parquet: {load_time_parquet:.02f} sec")
data.head()

shape of data: (5689512, 61), load time for csv: 36.82 sec, load time for parquet: 4.65 sec


,FlightDate,Airline,Origin,Dest,Cancelled,Diverted,CRSDepTime,DepTime,DepDelayMinutes,DepDelay,...,WheelsOff,WheelsOn,TaxiIn,CRSArrTime,ArrDelay,ArrDel15,ArrivalDelayGroups,ArrTimeBlk,DistanceGroup,DivAirportLandings
0,2018-01-23,Endeavor Air Inc.,ABY,ATL,False,False,1202,1157.0,0.0,-5.0,...,1211.0,1249.0,7.0,1304,-8.0,0.0,-1.0,1300-1359,1,0.0
1,2018-01-24,Endeavor Air Inc.,ABY,ATL,False,False,1202,1157.0,0.0,-5.0,...,1210.0,1246.0,12.0,1304,-6.0,0.0,-1.0,1300-1359,1,0.0
2,2018-01-25,Endeavor Air Inc.,ABY,ATL,False,False,1202,1153.0,0.0,-9.0,...,1211.0,1251.0,11.0,1304,-2.0,0.0,-1.0,1300-1359,1,0.0
3,2018-01-26,Endeavor Air Inc.,ABY,ATL,False,False,1202,1150.0,0.0,-12.0,...,1207.0,1242.0,11.0,1304,-11.0,0.0,-1.0,1300-1359,1,0.0
4,2018-01-27,Endeavor Air Inc.,ABY,ATL,False,False,1400,1355.0,0.0,-5.0,...,1412.0,1448.0,11.0,1500,-1.0,0.0,-1.0,1500-1559,1,0.0


Part 2:

## Cleanup
---
Let's clean our data in this following part : 

In [ ]:
# We are only interessted in flights that actually landed at the correct destination
data = data[(data['Cancelled'] == 0) & (data['Diverted'] == 0)]
data = data.drop(columns=['Cancelled', 'Diverted'])


# We're only keeping essential features :
cols_to_keep = [
    'Month', 'DayofMonth', 'DayOfWeek', 
    'Airline', 'Origin', 'Dest', 
    'CRSDepTime', 'Distance', 'DepDelay', 'ArrDelay'
]
data = data[cols_to_keep]

# Removing lines where the "ArrDelay" is empty/missing
data = data.dropna(subset=['ArrDelay']) # dropna = drop Not a Number (empty/missing values).


# Engineering des heures (Transformer HHMM en heures décimales toujours en base 24h)
# Exemple : 1530 devient 15.5 heures
"""data['DepHours'] = round(((data['CRSDepTime'] // 100) + (data['CRSDepTime'] %100)/60),2)""" 
# Temps tout en minutes (1202 devient 722 minutes)
data['DepHours'] = round(data['CRSDepTime'] // 100) * 60 + round((data['CRSDepTime'] % 100) / 60*60 )

data = data.drop(columns=['CRSDepTime'])


# Adding a binary feature (0,1) to indicate if there is a delay
data['IsDelay'] = (data['ArrDelay'] > 15).astype(int)

# Transforming text collums into categorys for memory optimisation
for col in ['Airline', 'Origin', 'Dest']:
    data[col] = data[col].astype('category')

print(data.shape)
print(data.head(10))

(5586619, 12)
    Month  DayofMonth  DayOfWeek            Airline Origin Dest  CRSDepTime  \
0       1          23          2  Endeavor Air Inc.    ABY  ATL        1202   
1       1          24          3  Endeavor Air Inc.    ABY  ATL        1202   
2       1          25          4  Endeavor Air Inc.    ABY  ATL        1202   
3       1          26          5  Endeavor Air Inc.    ABY  ATL        1202   
4       1          27          6  Endeavor Air Inc.    ABY  ATL        1400   
5       1          28          7  Endeavor Air Inc.    ABY  ATL        1202   
6       1          29          1  Endeavor Air Inc.    ABY  ATL        1202   
7       1          30          2  Endeavor Air Inc.    ABY  ATL        1202   
9       1           3          3  Endeavor Air Inc.    ATL  ABY        1037   
10      1           4          4  Endeavor Air Inc.    ATL  ABY        1037   

    Distance  DepDelay  ArrDelay  DepHours  IsDelay  
0      145.0      -5.0      -8.0     722.0        0  
1      1

Part 3:

## EDA : 
---

In [ ]:
plt.figure(figsize=(12, 6))

hourly_stats = data.groupby('DepHour')['IsDelay'].mean()
hourly_stats.plot(kind='bar', color='skyblue')

plt.title("Probabilité de retard (>15min) selon l'heure de départ")
plt.xlabel("Heure de la journée (0-23h)")
plt.ylabel("% de vols en retard")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

LINEAR REGRESSION :